# 03 — Tratamento e engenharia de atributos

Cria a camada Silver, a variável-alvo `acidente_grave`, atributos temporais e flags para o campo multivalorado `tracado_via`.

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np

PASTA_PROJETO = "TCC_PRF"
PASTA_PROCESSED = os.path.join(PASTA_PROJETO, "data", "processed")
df = pd.read_parquet(os.path.join(PASTA_PROCESSED, "prf_acidentes_2020_2025.parquet"))
silver = df.copy()

In [ ]:
colunas_texto = silver.select_dtypes(include=["object", "string"]).columns
for c in colunas_texto:
    silver[c] = silver[c].astype("string").str.strip()
    silver[c] = silver[c].replace(["", "nan", "None", "NULL", "null", "<NA>"], pd.NA)

silver["data_inversa"] = pd.to_datetime(silver["data_inversa"], errors="coerce")
silver["ano"] = silver["data_inversa"].dt.year
silver["mes"] = silver["data_inversa"].dt.month
silver["trimestre"] = silver["data_inversa"].dt.quarter
silver["dia_mes"] = silver["data_inversa"].dt.day
silver["dia_ano"] = silver["data_inversa"].dt.dayofyear
silver["fim_semana"] = silver["data_inversa"].dt.dayofweek.isin([5, 6]).astype("int8")

horario_convertido = pd.to_datetime(silver["horario"], format="%H:%M:%S", errors="coerce")
silver["hora"] = horario_convertido.dt.hour
silver["minuto"] = horario_convertido.dt.minute

def classificar_faixa_horaria(h):
    if pd.isna(h): return "Não informado"
    if h < 6: return "Madrugada"
    if h < 12: return "Manhã"
    if h < 18: return "Tarde"
    return "Noite"

silver["faixa_horaria"] = silver["hora"].apply(classificar_faixa_horaria)

In [ ]:
for c in ["br","km","pessoas","mortos","feridos_leves","feridos_graves","ilesos","ignorados","feridos","veiculos","latitude","longitude"]:
    silver[c] = pd.to_numeric(silver[c], errors="coerce")

for c in ["tipo_acidente","classificacao_acidente","regional","delegacia","uop"]:
    silver[c] = silver[c].fillna("Não informado")

silver["acidente_grave"] = ((silver["mortos"] > 0) | (silver["feridos_graves"] > 0)).astype("int8")
silver["acidente_fatal"] = (silver["mortos"] > 0).astype("int8")

In [ ]:
tracado_dummies = silver["tracado_via"].fillna("").str.get_dummies(sep=";")

def normalizar_nome_coluna(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = texto.encode("ascii", "ignore").decode("ascii").lower()
    texto = re.sub(r"[^a-z0-9]+", "_", texto).strip("_")
    return texto

tracado_dummies.columns = [f"tracado_{normalizar_nome_coluna(c)}" for c in tracado_dummies.columns]
silver = pd.concat([silver, tracado_dummies.astype("int8")], axis=1)

colunas_tracado = [c for c in silver.columns if c.startswith("tracado_") and c != "tracado_via"]
silver["qtd_caracteristicas_tracado"] = silver[colunas_tracado].sum(axis=1)

print(silver["acidente_grave"].value_counts())
print("IDs duplicados:", silver["id"].duplicated().sum())
print("Inconsistências de ano:", (silver["ano"] != silver["ano_fonte"]).sum())

In [ ]:
arquivo_silver = os.path.join(PASTA_PROCESSED, "prf_acidentes_silver_2020_2025.parquet")
silver.to_parquet(arquivo_silver, index=False)
print(f"Silver salva: {silver.shape[0]:,} registros e {silver.shape[1]} colunas")